# Interferometric X-Band Observations of the Sun

## AY 121 - Lab 03: Radio Interferometry

This is the **theory/support notebook** for the later analysis notebooks, not a standalone paper. Its role is narrower: derive only the equations that notebooks 03-05 actually use, define the symbols, and pin down the lab-manual fitting conventions.

What this notebook supplies to the rest of Lab 03:

1. the geometric delay and point-source fringe equation;
2. the fringe-frequency formula used for Fourier predictions and STFT fitting;
3. the uniform-disk / Bessel visibility used for the solar-diameter measurement;
4. the mapping between the lab-manual $(Q_{\mathrm{ew}}, Q_{\mathrm{ns}})$ notation and the notebook implementations.

A baseline prior is available from a direct in-situ lidar measurement of the antenna phase-centre separation:

$$b_{\mathrm{ew}}^{\mathrm{lidar}} = 15.17 \pm 0.30\;\mathrm{m},\qquad b_{\mathrm{ns}}^{\mathrm{lidar}} = 1.36 \pm 0.30\;\mathrm{m}. $$

The 30 cm 1-sigma uncertainty is a deliberately conservative floor that combines the single-shot lidar accuracy at ~15 m range with the unmodelled contribution from identifying the antenna phase centre by eye on a real dish. These values are stored in `utils/constants.py` and reappear later as the external sanity check on the radio-derived baseline.


## Scientific roadmap

### What we want to know

This lab uses a two-element east–west radio interferometer to answer **three concrete questions** about the Sun and our own instrument:

1. **How big is the Sun at 10 GHz?** The optical photospheric diameter is $\theta_\odot^{\mathrm{opt}} \approx 31.6'$ (operational value from [AY121-Lab3]; physical mean $\sim 31.99'$ from [BCD98]), but the centimetre Sun is bigger because the radio emission comes from the chromosphere and low corona, a few thousand km above the photosphere ([Stix] §10.3, [Dulk85] §III). The closest *measured* published radio radius is the NoRH 17 GHz value of [Selhorst04]: $\theta_\odot(17\,\mathrm{GHz}) = 32.55 \pm 0.05'$ (mean over 3800+ NoRH maps spanning 1992–2003); 10 GHz is expected to be *slightly larger* (longer wavelength → higher chromosphere), so the comparison value for our observation is $\sim 32.6'$. Can we measure that $\sim 1\,\%$ excess directly from a single afternoon of fringes? *(Answered in notebook 05.)*
2. **Is there an active region on the visible solar disk today, and where is it?** A localised bright (or dark) feature on top of the disk produces a non-zero residual visibility at the Bessel nulls of the uniform-disk model — both an *amplitude* signature (flux fraction $f$) and a *phase* signature (EW offset $\Delta\alpha$). Cross-checking against the NOAA SWPC / SDO HMI active-region catalogue for the observation date converts a fringe wiggle into a piece of solar physics. *(Answered in notebook 05.)*
3. **What is our array geometry, and can the fringes themselves measure it?** We have a direct in-situ lidar measurement of the antenna phase-centre separation, $b_{\mathrm{ew}} = 15.17 \pm 0.30\;\mathrm{m}$ and $b_{\mathrm{ns}} = 1.36 \pm 0.30\;\mathrm{m}$. Can we recover those numbers from the fringe data alone (a meta-result about the *technique*)? *(Answered in notebook 04.)*

These three questions are answered by three independent measurements that share one calibrated dataset.

### How the notebooks fit together

```
   ┌────────────────────────────────┐
   │ 01  Theory                     │   (you are here)
   │     - geometric delay          │
   │     - fringe equation          │
   │     - VC–Z + Bessel envelope   │
   │     - 5 baseline methods       │
   │     - sunspot diagnostics      │
   └───────────────┬────────────────┘
                   │
   ┌───────────────▼────────────────┐
   │ 02a / 02b  Data inspection     │   "what is the signal,
   │     - raw V(ν, h)              │    what does our cleanup
   │     - chip-to-chip gain check  │    do to it?"
   │     - per-capture σ_V          │
   │     - adaptive DC correction   │
   │     - synthetic injection test │
   └───────────────┬────────────────┘
                   │
   ┌───────────────▼────────────────┐
   │ 03  Phenomenological models    │   "what does the lidar
   │     - predict fringe period    │    prior predict, and
   │     - predict Bessel envelope  │    where do we test it?"
   │     - lidar-prior overlay      │
   └───────────────┬────────────────┘
                   │
   ┌───────────────▼────────────────┐   ┌────────────────────────┐
   │ 04  Baseline determination     │   │ Question 3 answered:   │
   │     - 5 independent methods    │──▶│ array geometry from    │
   │     - lidar cross-check        │   │ fringes vs lidar       │
   │     - calibrated u-axis        │   └────────────────────────┘
   └───────────────┬────────────────┘
                   │ (calibration of the u-axis)
                   │
   ┌───────────────▼────────────────┐   ┌────────────────────────┐
   │ 05  Solar science              │   │ Questions 1 + 2:       │
   │     - Bessel envelope fit      │──▶│ radio diameter +       │
   │     - sunspot amplitude + phase│   │ active region (Δα, f)  │
   │     - SDO/HMI cross-check      │   └────────────────────────┘
   └────────────────────────────────┘
```

The right-hand column lists the *scientific question each notebook answers*. Notebooks 02–04 are setup; **the science is in notebook 05**, and the synthesis at the end of nb 05 ties all three results together.

---

## 1.1 The Adding Interferometer and Path Delay

Consider two antennas separated by a baseline vector $\mathbf{b}$ observing a distant point source. The electric field at antenna 1 is

$$E_1(t) = E_0 \cos(2\pi\nu\, t),$$

while at antenna 2 the signal arrives with a geometric time delay $\tau_g$:

$$E_2(t) = E_0 \cos\!\bigl[2\pi\nu\,(t + \tau_{\mathrm{tot}})\bigr],$$

where $\tau_{\mathrm{tot}} = \tau_g + \tau_c$ is the total delay, comprising the geometric delay $\tau_g$ (which depends on source position and baseline geometry) and a constant instrumental/cable delay $\tau_c$.

### Geometric delay for an EW + NS baseline

For an interferometer at geographic latitude $L$ with east–west baseline component $b_{\mathrm{ew}}$ and north–south component $b_{\mathrm{ns}}$, the geometric delay for a source at declination $\delta$ and hour angle $h$ is

$$\boxed{\tau_g(h) = \frac{b_{\mathrm{ew}}}{c}\cos\delta\,\sin h + \frac{b_{\mathrm{ns}}}{c}\sin L\,\cos\delta\,\cos h.}$$

(Standard rotation from the local topocentric $(E, N, U)$ frame to the equatorial $(X, Y, Z)$ frame; see [TMS3] eqs. 4.1–4.4 and the worked derivation in [TMS3] §4.1.)

**Sign convention.** $\tau_g > 0$ means the wavefront reaches antenna 1 before antenna 2; equivalently, $\tau_g = \mathbf{b}_{21}\cdot\hat{s}/c$ where $\mathbf{b}_{21}$ points *from* antenna 2 (east/north) *to* antenna 1 (west/south). In particular, when the source is rising ($h < 0$, east of the meridian), the wavefront reaches the east antenna first, giving $\tau_g < 0$.

**Derivation.** The baseline vector in the local topocentric frame is $\mathbf{b} = (b_{\mathrm{ew}},\, b_{\mathrm{ns}},\, 0)$ (east, north, up). The unit vector toward the source in equatorial coordinates is $\hat{s} = (\cos\delta\cos h,\;\cos\delta\sin h,\;\sin\delta)$. Projecting $\hat{s}$ into the topocentric frame via the rotation matrix $R(\text{eq}\to\text{topo})$ that accounts for the observatory latitude $L$, the delay is $\tau_g = \mathbf{b}\cdot\hat{s}_{\mathrm{topo}}/c$. Carrying out the matrix multiplication yields the expression above; see [TMS3] §4.1 for the explicit rotation.

The north–south contribution has two terms: a $\cos h$ term (time-varying) and a constant term $-(b_{\mathrm{ns}}/c)\cos L\,\sin\delta$ that is absorbed into an effective cable delay:

$$\tau'_c = \tau_c - \frac{b_{\mathrm{ns}}}{c}\cos L\,\sin\delta.$$

## 1.2 The Fringe Pattern (Point Source)

The NCH interferometer is an **adding** interferometer (after [Ryle52]): it combines the voltages from the two antennas, passes the sum through a square-law (power) detector, and time-averages to suppress the radio-frequency oscillation. The detected power is

$$P(t) = \bigl[E_1(t) + E_2(t)\bigr]^2 = E_0^2\cos^2(2\pi\nu t) + 2E_0^2\cos(2\pi\nu t)\cos\!\bigl[2\pi\nu(t+\tau_{\mathrm{tot}})\bigr] + E_0^2\cos^2\!\bigl[2\pi\nu(t+\tau_{\mathrm{tot}})\bigr].$$

Expanding the cross-term with the product-to-sum identity and time-averaging to remove all terms oscillating at $2\nu$:

$$\langle P \rangle = \underbrace{E_0^2}_{\text{DC (total power)}} + \underbrace{E_0^2\cos\!\bigl[2\pi\nu\,\tau_{\mathrm{tot}}(h)\bigr]}_{\text{fringe}}.$$

The first term is the sum of the two individual antenna powers — a slowly-varying DC baseline that is removed in post-processing (Section 2). The oscillating cross-term is the **fringe**, carrying all geometric information. After DC removal:

$$F(h) = E_0^2 \cos\!\bigl[2\pi\nu\,\tau_{\mathrm{tot}}(h)\bigr].$$

> **Note (correlating interferometer).** A correlating interferometer computes $\langle E_1 E_2\rangle$ directly, yielding $\tfrac{E_0^2}{2}\cos(2\pi\nu\tau_{\mathrm{tot}})$ — the same fringe phase and period, without a DC offset and with half the amplitude. Both architectures encode identical geometric information; the adding interferometer is simpler to build at the cost of requiring DC removal. See [TMS3] §1.3 for the historical comparison and [Ryle52] for the original adding-interferometer paper.

Expanding the total delay $\tau_{\mathrm{tot}} = \tau'_g(h) + \tau'_c$ and absorbing the (unknown) cable-delay phase $2\pi\nu\tau'_c$ into two fitting constants $A$ and $B$:

$$\boxed{F(h) = A\cos\!\bigl(2\pi\nu\,\tau'_g(h)\bigr) + B\sin\!\bigl(2\pi\nu\,\tau'_g(h)\bigr),}$$

where $\tau'_g(h) = (b_{\mathrm{ew}}/c)\cos\delta\,\sin h + (b_{\mathrm{ns}}/c)\sin L\,\cos\delta\,\cos h$ is the time-varying part of the geometric delay (the effective geometric delay). The constants satisfy $A = E_0^2\cos(2\pi\nu\tau'_c)$ and $B = -E_0^2\sin(2\pi\nu\tau'_c)$, so the total amplitude is $\sqrt{A^2 + B^2} = E_0^2$ and the phase offset is $\phi_0 = \arctan(-B/A) = 2\pi\nu\tau'_c$.

This is the fundamental equation of the adding interferometer: for a point source, the fringe is a quasi-sinusoidal function of hour angle whose argument depends on the baseline components and source declination.

> **Caveats — what is hidden in $A$, $B$, and $\tau'_c$.** The treatment above assumes a single, time-invariant cable phase and a single per-antenna voltage gain $E_0$ that is identical for both antennas and constant across the analysis band. In practice each receiver has its own complex gain $g_i(\nu, t) = |g_i(\nu)|\,e^{i\phi_i(\nu, t)}$ that drifts with temperature, has frequency structure (bandpass), and may shift between observing "chips" (segments separated by retunes or restarts of the back-end). The boxed equation should therefore be read as the **per-channel, per-chip** model: $A$ and $B$ are local fitting constants, not global ones, and any analysis that pools data across chips must either solve for, or remove, the chip-to-chip gain and phase offsets first. ([TMS3] §10 and §11 discuss the standard CLEAN/self-calibration approaches that lift these restrictions for production interferometers.)
>
> **Bandwidth and time decorrelation.** Two further effects are small but not zero. (i) *Bandwidth decorrelation* multiplies $F$ by $\mathrm{sinc}(\pi\,\Delta\nu\,\tau'_g)$, where $\Delta\nu$ is the channel width ([TMS3] §6.3, eq. 6.61). With $\Delta\nu = F_S/N_{\mathrm{FFT}} = 244\;\mathrm{kHz}$ and $|\tau'_g| \lesssim 50\;\mathrm{ns}$ (set by the maximum delay $b_{\mathrm{ew}}/c \approx 50\;\mathrm{ns}$), the argument $\pi\,\Delta\nu\,\tau'_g \lesssim 0.038$, giving a per-channel loss of $\sim (\pi\Delta\nu\tau)^2/6 \approx 2.4\times 10^{-4}$ — completely ignorable. The envelope tilt across the 70 MHz analysis band, $\sim (\pi\,\Delta\nu_{\mathrm{band}}\tau)^2/6$ with $\Delta\nu_{\mathrm{band}} = 70\;\mathrm{MHz}$, is $\sim 10\%$ at the longest delays — *not* negligible at the band-averaged level if pooled across many captures, which is why we work per-channel. (ii) *Time-average smearing*: integrating for $\Delta t$ multiplies by $\mathrm{sinc}(\pi f_f\,\Delta t)$. With $\Delta t \approx 2.5\;\mathrm{s}$ and $|f_f| \lesssim 0.05\;\mathrm{Hz}$ at transit, the argument $\pi\,f_f\,\Delta t \lesssim 0.39$, giving $\mathrm{sinc} \approx 0.974$, i.e. a loss of $\sim 2.5\,\%$ at transit and smaller toward the horizon — small enough to fold into the noise budget but worth flagging.

### Optional intuition: physical analogies for the fringe

**The fringe as a "giant DSB mixer in the sky."**
The adding interferometer's power output contains a term $\cos(2\pi\nu\tau_g)$, which is formally identical to the output of a **double-sideband (DSB) mixer**. In a DSB mixer, a signal at frequency $\nu_{\mathrm{sig}}$ is multiplied by a local oscillator at $\nu_{\mathrm{LO}}$, producing sum and difference tones. Here the sky signal arrives at the two antennas with a relative geometric delay $\tau_g(h)$ that changes as the Earth rotates - the delay acts as a continuously-tuned "LO phase." The interferometer output oscillates at the *fringe frequency* $f_f = d\phi/dt = \nu\,d\tau_g/dt$, which plays the role of the difference tone. Just as a DSB mixer is sensitive to signals at both $\nu_{\mathrm{LO}} \pm \nu_{\mathrm{IF}}$, the adding interferometer responds equally to sources on either side of the fringe null - the well-known **fringe ambiguity** of a single-baseline measurement.

**The fringe as a correlation function.**
Equivalently, the fringe can be understood as the **mutual coherence function** of the two antenna voltages:

$$V_{12}(\tau) \;=\; \langle\, v_1(t)\; v_2^*(t - \tau)\,\rangle.$$

For a point source at geometric delay $\tau_g$, this reduces to $V_{12} \propto e^{i2\pi\nu\tau_g}$ - the cross-correlation picks out the single delay at which the signals are coherent. For an *extended* source, the sum over all source directions produces a weighted integral of $e^{i2\pi\nu\tau_g(\theta)}$, which is precisely the Van Cittert-Zernike theorem derived in Section 1.4. The fringe visibility is thus the Fourier transform of the sky brightness distribution, sampled at the spatial frequency set by the projected baseline.


## 1.2b The lab-manual form: $Q_{\mathrm{ew}}$, $Q_{\mathrm{ns}}$, and the least-squares problem

The AY 121 lab manual [`src/ugradio/lab_interf/interf.tex`](../../../src/ugradio/lab_interf/interf.tex) (§4, §5) writes the point-source fringe in a form that makes explicit which parameters are linear and which are nonlinear in the least-squares fit. Starting from the delay decomposition of §1.1 and the boxed fringe equation of §1.2,

$$F(h_s) \;=\; A\cos\!\bigl(2\pi\nu\,\tau'_g(h_s)\bigr) + B\sin\!\bigl(2\pi\nu\,\tau'_g(h_s)\bigr),$$

the lab manual absorbs the observing frequency into two dimensionless parameters, measured in **wavelengths**:

$$\boxed{\,Q_{\mathrm{ew}} \;\equiv\; \frac{b_{\mathrm{ew}}}{\lambda}\cos\delta,\qquad Q_{\mathrm{ns}} \;\equiv\; \frac{b_{\mathrm{ns}}}{\lambda}\sin L\,\cos\delta.\,}$$

(There is one subtlety of notation: some lab-manual passages treat $Q_{\mathrm{ew}}$ and $Q_{\mathrm{ns}}$ as the *product* with $\cos\delta$, others as $b/\lambda$ alone. The implementation in `utils/baseline_fitting.py` scans over $(b_{\mathrm{ew}}/\lambda, b_{\mathrm{ns}}/\lambda)$ as the nonlinear unknowns and folds per-capture $\cos\delta_i$ into the phase computation, so the final answer does not depend on the convention; see the docstring of `grid_search_baseline`.)

In these variables the fringe phase at capture $i$ is

$$\psi_i \;=\; 2\pi\bigl[Q_{\mathrm{ew}}\sin h_i + Q_{\mathrm{ns}}\cos h_i\bigr],$$

and the lab-manual form of the least-squares problem is: for each guessed pair $(Q_{\mathrm{ew}}, Q_{\mathrm{ns}})$ the linear coefficients $(A, B)$ are solved analytically by **projection** (separable variable least squares [Golub & Pereyra 1973, SIAM J. Numer. Anal. 10, 413]), and the residual sum of squares $\mathcal{S}^2(Q_{\mathrm{ew}}, Q_{\mathrm{ns}})$ is tabulated. The minimum of $\mathcal{S}^2$ locates the baseline; the curvature of $\mathcal{S}^2$ at the minimum gives the covariance. This is the procedure spelled out step-by-step in [`fitting_notes_2017.pdf`](../../../src/ugradio/lab_interf/fitting_notes_2017.pdf):

1. **1-D preliminary sweep** of $\mathcal{S}^2$ vs $Q_{\mathrm{ew}}$ with $Q_{\mathrm{ns}}=0$, over a wide, finely-sampled range, to locate the 1-D minimum and confirm that it is not a sidelobe.
2. **2-D sweep** over both $(Q_{\mathrm{ew}}, Q_{\mathrm{ns}})$ to extract the global 2-D minimum $(Q_{\mathrm{ew}}^\star, Q_{\mathrm{ns}}^\star)$.
3. **Curvature matrix** $[\alpha]$ assembled numerically from the 2-D $\Delta\mathcal{S}^2$ grid (second derivatives at the minimum).
4. **Covariance** $\mathrm{cov} = [\alpha]^{-1}$ (scaled by the per-point variance estimate $\sigma^2 \approx \mathcal{S}^2_{\min}/\mathrm{dof}$ when $\sigma$ is not known *a priori*).
5. **Uncertainties** $\sigma_{Q_{\mathrm{ew}}} = \sqrt{\mathrm{cov}_{11}}$, $\sigma_{Q_{\mathrm{ns}}} = \sqrt{\mathrm{cov}_{22}}$ (lsfit-lite eq. 3.7).

Notebook 04 walks through this chain end-to-end, produces the required plots (1-D sweep, 2-D contour, $\Delta\mathcal{S}^2$ with $1$/$2\sigma$ confidence ellipses), and compares against a "proper" Levenberg–Marquardt nonlinear least-squares fit seeded from the brute-force best-fit — the "if you have time, try both methods" comparison the lab manual explicitly calls for in §5.2.

The same $(Q_{\mathrm{ew}}, Q_{\mathrm{ns}})$ parameters recur in §1.3 below as the coefficients of the fringe-frequency formula: no coincidence, because the fringe frequency is the time derivative of the fringe phase $\psi(h)$.

## 1.3 Fringe Frequency

The fringe phase $\phi(h) = 2\pi\nu\,\tau'_g(h)$ changes as the Earth rotates the source through the fringe pattern. The instantaneous fringe frequency (in Hz) is

$$f_f = \frac{1}{2\pi}\frac{d\phi}{dt} = \nu\,\frac{d\tau'_g}{dt}.$$

Since $h$ changes at the sidereal rate $\omega_\oplus = 2\pi / T_{\mathrm{sid}} \approx 7.292115\times10^{-5}\;\mathrm{rad\,s^{-1}}$ (with $T_{\mathrm{sid}} = 86\,164.0905\;\mathrm{s}$, [IERS2010]), differentiating $\tau'_g$ with respect to $h$ and multiplying by $\omega_\oplus$:

$$\boxed{f_f(h) = \omega_\oplus\left[\frac{b_{\mathrm{ew}}}{\lambda}\cos\delta\,\cos h - \frac{b_{\mathrm{ns}}}{\lambda}\sin L\,\cos\delta\,\sin h\right],}$$

where $\lambda = c/\nu$ is the observing wavelength (with $c = 299\,792\,458\;\mathrm{m\,s^{-1}}$ defined exactly by the SI, [BIPM-SI]).

### Fringe period at the meridian

At transit ($h = 0$) with $b_{\mathrm{ns}} \approx 0$:

$$P_f = \frac{1}{|f_f|} = \frac{\lambda}{\omega_\oplus\, b_{\mathrm{ew}}\,\cos\delta} \approx \frac{26\;\mathrm{s}}{\cos\delta}$$

for $b_{\mathrm{ew}} \approx 15.17\;\mathrm{m}$ (lidar prior, see `constants.py`) and $\lambda \approx 2.87\;\mathrm{cm}$ at $10.45\;\mathrm{GHz}$ ($P_f = 0.02868\,\mathrm{m} / (7.292\times10^{-5}\,\mathrm{rad\,s^{-1}} \times 15.17\,\mathrm{m}) \approx 25.9\;\mathrm{s}$). The fringe period is **shortest at transit** (where $|\cos h| = 1$) and stretches toward infinity at the horizon ($|\cos h| \to 0$).

### The x-coordinate linearisation

The fringe phase is $\phi = 2\pi(b_{\mathrm{ew}}/\lambda)\,x$ where $x \equiv \cos\delta\,\sin h$. In this coordinate the fringe is a **pure sinusoid**, which means an FFT in $x$-space directly yields the spatial frequency $b_{\mathrm{ew}}/\lambda$. This is the basis of the FFT baseline method (Section 3.1; the same trick is described in [TMS3] §10.4 in the context of single-baseline calibration).

### Delay vs sky-plane baseline — a critical distinction

Two different projections of the baseline arise naturally:

| Quantity | Formula (EW baseline, $b_{\mathrm{ns}} = 0$) | At transit | At horizon |
|----------|-------|------------|------------|
| **Delay baseline** $w = \nu\tau_g$ | $(b_{\mathrm{ew}}/\lambda)\cos\delta\,\sin h$ | **0** | **max** |
| **Sky-plane baseline** $\|u_{\mathrm{sky}}\|$ | $(b_{\mathrm{ew}}/\lambda)\cos\delta\,\|\cos h\|$ (small-$\delta$ approx.) | **max** | **0** |

These are **complementary in the equatorial plane**: $w^2 + u_{\mathrm{sky}}^2 = (b_{\mathrm{ew}}\cos\delta/\lambda)^2$ when $\sin\delta = 0$.

> **More carefully.** The exact relation in $(u, v, w)$ ([TMS3] eq. 4.5) is $u^2 + v^2 + w^2 = |b|^2/\lambda^2$. For a pure EW baseline at hour angle $h$, declination $\delta$, the standard expressions are
> $$u = (b_{\mathrm{ew}}/\lambda)\cos h, \quad v = (b_{\mathrm{ew}}/\lambda)\sin\delta\,\sin h, \quad w = -(b_{\mathrm{ew}}/\lambda)\cos\delta\,\sin h,$$
> so the **true** sky-plane magnitude is $\sqrt{u^2+v^2} = (b_{\mathrm{ew}}/\lambda)\sqrt{\cos^2 h + \sin^2\delta\,\sin^2 h}$. The simpler form $|u_{\mathrm{sky}}| = (b_{\mathrm{ew}}/\lambda)\cos\delta\,|\cos h|$ used in the table is the **small-$\delta$ approximation** valid when $\sin^2\delta \ll 1$. For the Sun in the present dataset $\delta \approx -0.08^\circ$, so the approximation is exact at the $10^{-6}$ level and is what is used in the implementation. The table identity $w^2 + u_{\mathrm{sky}}^2 = (b_{\mathrm{ew}}\cos\delta/\lambda)^2$ should therefore be read as a small-$\delta$ statement, not a general identity.

- The **delay** $w$ determines the fringe *phase* (path difference between antennas). It is zero at transit because the source is perpendicular to the baseline.
- The **sky-plane baseline** $|u_{\mathrm{sky}}|$ determines the *spatial resolution* — how many fringe cycles fit across the source. It is **maximum at transit** (the full baseline is projected onto the sky) and zero at the horizon (the baseline is parallel to the line of sight).

The Bessel-function modulation of an extended source (Section 1.5) depends on $|u_{\mathrm{sky}}|$, **not** on $w$.

> **Sidereal vs solar rate.**  The hour angle $h$ tracks *sidereal* time, so the angular rate $\omega_\oplus = 2\pi / T_{\mathrm{sidereal}} = 7.2921 \times 10^{-5}\;\mathrm{rad\,s^{-1}}$ (IERS 2010) rather than $2\pi / T_{\mathrm{solar}}$. The 0.27% difference matters at the precision level of this lab — using the solar-day rate would shift the derived baseline by $\sim 0.27\%\times 15\;\mathrm{m} \approx 4\;\mathrm{cm}$, which exceeds our quoted uncertainty.

## 1.4 Extended Sources and the Van Cittert–Zernike Theorem

For an extended source with one-dimensional brightness distribution $I(\theta)$ (brightness as a function of angular offset $\theta$ in radians from the phase centre), the interferometer response is the **convolution** of the point-source fringe pattern with the source brightness. In the visibility domain this becomes a multiplication:

$$R(h) = F(h) \times \underbrace{\int_{-\infty}^{\infty} I(\theta)\,e^{-2\pi i\,|u_{\mathrm{sky}}|\,\theta}\,d\theta}_{\displaystyle V(|u_{\mathrm{sky}}|)\;\text{(complex visibility)}}.$$

This is the one-dimensional form of the **van Cittert–Zernike theorem** ([vC34], [Z38]; for the radio-astronomical statement and the extension to two dimensions and partially polarised sources see [TMS3] Ch. 14, eq. 14.7).

For a source brightness that is **real and symmetric** about the phase centre — as for a centred uniform disk — the imaginary part of the integral vanishes and the visibility reduces to a Fourier *cosine* transform,

$$V(|u_{\mathrm{sky}}|) \;=\; \int_{-\infty}^{\infty} I(\theta)\,\cos\!\bigl(2\pi\,|u_{\mathrm{sky}}|\,\theta\bigr)\,d\theta \;\equiv\; \mathrm{MF}(|u_{\mathrm{sky}}|).$$

The **modulating factor** $\mathrm{MF}(|u_{\mathrm{sky}}|)$ is therefore the Fourier transform of the source brightness, evaluated at the instantaneous sky-plane baseline $|u_{\mathrm{sky}}|$ (in wavelengths). The argument $2\pi|u_{\mathrm{sky}}|\,\theta$ is dimensionless (wavelengths $\times$ radians), as required. Since $|u_{\mathrm{sky}}| = |f_f|/\omega_\oplus$ (see Section 1.3), this is equivalent to taking the Fourier transform at fringe frequency $f_f$ with $\theta$ expressed in time units via $\theta_{\mathrm{time}} = \theta/\omega_\oplus$.

> **Why we keep the cosine form for the disk and the full complex form for the spot.** When the source brightness is *not* symmetric about the phase centre — for example, the disk plus an off-axis sunspot — the imaginary part of the integral does *not* vanish, and the modulating factor becomes complex. A point source displaced by $\Delta\alpha$ contributes a visibility $\propto e^{-2\pi i\,u\,\Delta\alpha}$ — a pure phase whose slope encodes the offset (used in §1.7 to localise sunspots). For the centred uniform disk treated in §1.5 the cosine form is exact; for §1.7 we keep the full complex form.

**Physical intuition:** when the fringe spacing $\lambda/|u_{\mathrm{sky}}|$ is much larger than the source, fringe peaks and troughs cover the source uniformly and the response is just the point-source fringe scaled by total flux. When the fringe spacing equals the source size, equal amounts of the source fall on positive and negative fringe lobes, and the response **cancels** — producing a null in the fringe amplitude envelope.

## 1.5 The Uniform Disk: Bessel-Function Visibility

### Brightness distribution

Model the Sun as a uniformly bright circular disk of angular radius $R$. For a 1-D cut through the disk centre, the chord length at offset $\theta$ is proportional to $\sqrt{R^2 - \theta^2}$, giving the projected brightness profile:

$$I(\theta) = \begin{cases} \displaystyle\frac{2}{\pi R^2}\sqrt{R^2 - \theta^2} & |\theta| < R \\[6pt] 0 & \text{otherwise}. \end{cases}$$

This is normalised to unit total flux: $\int_{-R}^{R} I(\theta)\,d\theta = 1$. Equivalently it is the projection along $v$ of a uniform 2-D disk, and by the Fourier slice theorem its 1-D Fourier transform is the same jinc as the 2-D Hankel transform of the disk ([B&W] §8.5, eq. 8.5.20).

### Visibility function

The Fourier transform of a uniform disk is a standard result in diffraction theory ([B&W] §8.5; [TMS3] §13.1, eq. 13.10). Substituting $I(\theta)$ into the MF integral (Section 1.4) and using the identity $\int_{-R}^{R}\sqrt{R^2-\theta^2}\,e^{-2\pi i u\theta}\,d\theta = \pi R^2 J_1(2\pi Ru)/(2\pi Ru)$, the normalised complex visibility is

$$\boxed{\frac{V(|u_{\mathrm{sky}}|)}{V(0)} = \frac{2\,J_1(2\pi\,|u_{\mathrm{sky}}|\, R)}{2\pi\,|u_{\mathrm{sky}}|\, R},}$$

where $J_1$ is the Bessel function of the first kind of order one ([DLMF] §10.2; [A&S] §9.1), and

$$|u_{\mathrm{sky}}| = \frac{|f_f|}{\omega_\oplus} = \left|\frac{b_{\mathrm{ew}}}{\lambda}\cos\delta\,\cos h - \frac{b_{\mathrm{ns}}}{\lambda}\sin L\,\cos\delta\,\sin h\right|$$

is the component of the baseline **perpendicular to the line of sight**, in units of wavelengths. This is the "jinc" function — the circular analogue of the sinc function.

**1-D projection approximation.** Strictly, the 2-D visibility of a circular disk is $2J_1(2\pi R\sqrt{u^2+v^2})/(2\pi R\sqrt{u^2+v^2})$, where $(u, v)$ are the full sky-plane baseline components (EW and NS). Here we use only $|u_{\mathrm{sky}}| \approx |u|$, dropping $v$. From §1.3, for a pure EW baseline $v = (b_{\mathrm{ew}}/\lambda)\sin\delta\sin h$. With $\delta \approx -0.08^\circ$ for the Sun in this dataset, $|v|/|u| < 0.0014\,|\tan h|$, i.e. $\lesssim 10^{-3}$ even at $|h| = 80^\circ$. The fractional bias on the inferred diameter is then $\lesssim (v/u)^2/2 \sim 10^{-6}$ — completely negligible. For an observation at a more inclined declination (e.g. a planet), this approximation would have to be revisited.

### Limb brightening at X-band and the limb-brightened visibility

The Sun's optical photospheric diameter at 1 AU has a mean value of $\sim 31.99'$ from helioseismic ([BCD98]) and direct-imaging ([Meftah18]) determinations; Earth's orbital eccentricity ($\sim 1.7\%$) modulates the apparent diameter from $\sim 31.46'$ at aphelion (early July) to $\sim 32.53'$ at perihelion (early January). The value $\theta_\odot^{\mathrm{opt}} \approx 31.6'$ used in `constants.py` (`SOLAR_DIAMETER_ARCMIN_NOMINAL = 31.6`) is the operational value from the AY 121 lab manual ([AY121-Lab3]), appropriate for an observation near aphelion. At centimetre wavelengths the situation is different: the dominant emission mechanism is *thermal free–free* in the chromosphere and low corona ([Dulk85] §III; [R&L] §5.2 for the underlying Bremsstrahlung treatment), whose temperature *increases* with height according to the semi-empirical chromospheric model of [VAL81] (and updates [FAL93], [FAL2009]):

$$T_e \approx 6\times 10^3\;\mathrm{K}\;\text{(photosphere)} \;\to\; \approx 2\times 10^4\;\mathrm{K}\;\text{(top of chromosphere)} \;\to\; \approx 10^6\;\mathrm{K}\;\text{(corona)}.$$

At 10 GHz the optical depth $\tau_\nu = 1$ surface lies a few thousand km above the photosphere, with brightness temperature $T_b \approx (1\text{–}2)\times 10^4\;\mathrm{K}$ ([Zirin91]; [Stix] §10.3), and the *limb* is mildly **brighter** than the disk centre because the slant path through the chromosphere is longer ([Dulk85] §III.B; [Selhorst-modelling] for explicit modelling at 10 GHz; [Alissandrakis-ALMA] for ALMA-era cm/mm observations). The net effect is twofold:

1. The **apparent radio radius** is larger than the optical radius by a few percent (textbook discussion: [Stix] §10.3, [Dulk85] §III). The closest *measured* published radio radius to our 10 GHz observation is the Nobeyama Radioheliograph value of [Selhorst04] at 17 GHz: $$R_\odot(17\,\mathrm{GHz}) = 976.6 \pm 1.5'' \quad\Longleftrightarrow\quad \theta_\odot(17\,\mathrm{GHz}) = 32.55 \pm 0.05'$$ (NoRH average over 1992–2003). The K-band measurements of [Marongiu24] at 18.3 / 25.8 GHz give $R_\odot \approx 976\text{–}982''$, consistent with this picture. Our observation is at 10 GHz, slightly *lower* in frequency than the Selhorst+04 anchor, so the radius is expected to be *slightly larger* than $976.6''$ — longer wavelength probes a higher chromospheric layer. We therefore use $\theta_\odot^{\mathrm{radio}}(10\,\mathrm{GHz}) \approx 32.6'$ as the comparison value, with the understanding that this is an extrapolation from a 17 GHz measurement and the lab is, in fact, *measuring* the radius at 10 GHz directly.
2. The **brightness profile is not flat**: cm-wavelength models of the quiet Sun show modest limb brightening over the outer few percent of the disk radius — see the Selhorst-Costa modelling series and the Alissandrakis ALMA papers for representative profiles.

#### Visibility of a limb-brightened disk

To model this, replace the uniform brightness $I_0$ with the simplest radial profile that preserves circular symmetry and peaks at the limb:

$$I(\rho) = I_0\!\left[1 + \varepsilon\!\left(\frac{\rho}{R}\right)^{\!2}\right], \qquad \rho \le R,$$

where $\varepsilon > 0$ parametrises the limb excess. The Hankel transform (§1.4 applied to a circularly symmetric source) splits into the uniform-disk jinc plus a quadratic correction:

$$V(q) = 2\pi I_0 \int_0^R \!\left[1 + \varepsilon\!\left(\frac{\rho}{R}\right)^{\!2}\right] J_0(2\pi q\rho)\,\rho\,d\rho = V_{\mathrm{disk}}(q) \;+\; \varepsilon\,\frac{2\pi I_0}{R^2}\int_0^R \rho^3\,J_0(2\pi q\rho)\,d\rho.$$

Substituting $t = \rho/R$ and defining $x = 2\pi q R$, the normalised visibility becomes

$$\boxed{V_{\mathrm{lb}}(q) = \frac{\dfrac{2\,J_1(x)}{x} \;+\; \varepsilon\cdot 2\!\displaystyle\int_0^1 t^3\,J_0(x\,t)\,dt}{1 + \varepsilon/2},}$$

where the denominator $1 + \varepsilon/2$ ensures $V_{\mathrm{lb}}(0) = 1$ (using $2J_1(x)/x \to 1$ and $2\int_0^1 t^3\,dt = 1/2$ as $x \to 0$). The quadratic integral is computed numerically in notebook 05 via a precomputed lookup table.

**Key properties.** (i) Setting $\varepsilon = 0$ recovers the uniform-disk jinc exactly. (ii) The $\int t^3 J_0(xt)\,dt$ term does **not** vanish at the jinc nulls ($J_1$ zeros), so a limb-brightened disk has non-zero visibility where a uniform disk has nulls — this is the source of the limb-brightening contamination of the sunspot flux fraction discussed in §1.7. (iii) The *positions* of the envelope extrema shift only weakly with $\varepsilon$ (sub-percent for $\varepsilon \lesssim 1$), which is why the geometric diameter measurement from extrema positions (notebook 05, Part II) is robust against limb-brightening uncertainty.

#### How this is used in the analysis

The uniform-disk jinc is the **starting point** for the lab-manual diameter measurement (§1.5b, §1.6): the extrema positions depend on $R$ through fixed Bessel roots. Notebook 05 extends this in two stages:

- **Part II** measures $R$ geometrically from the positions of envelope extrema (nulls at $J_1$ zeros, peaks at $J_2$ zeros). This is insensitive to the brightness profile because it uses only *where* features occur, not *how deep* or *how tall* they are.
- **Part III** then fixes $R$ and fits the full envelope shape to the limb-brightened model above, extracting $\varepsilon$ and the sunspot floor $f$. This decomposes the observed non-zero null depths into a disk-brightness component (limb brightening) and a point-source component (sunspot), which are degenerate at the nulls but have different $q$-dependence across the full envelope.

### Geometric intuition

At **transit** ($h = 0$), the EW baseline is perpendicular to the line of sight, so the full baseline is projected onto the sky: $|u_{\mathrm{sky}}|$ is **maximum**. Many fringe cycles span the solar disk, positive and negative contributions largely cancel, and the visibility is **low** (deep in the Bessel sidelobes).

At **sunrise/sunset** ($|h| \to 90°$), the baseline is nearly parallel to the line of sight, so $|u_{\mathrm{sky}}| \to 0$. The fringe spacing is much larger than the Sun, the entire disk contributes coherently, and the visibility is **maximum** ($\mathrm{jinc}(0) = 1$).

### Properties of the jinc function

Bessel-function zero values from [DLMF] Table 10.21.1 / [A&S] Table 9.5:

| Property | Value |
|----------|-------|
| $V(0)/V(0)$ | 1 (at horizon, where $\|u_{\mathrm{sky}}\| \to 0$) |
| First null | $2\pi\,\|u_{\mathrm{sky}}\|\,R = j_{1,1} \approx 3.8317$ |
| Second null | $2\pi\,\|u_{\mathrm{sky}}\|\,R = j_{1,2} \approx 7.0156$ |
| Third null | $2\pi\,\|u_{\mathrm{sky}}\|\,R = j_{1,3} \approx 10.1735$ |

The **fringe amplitude envelope** traces the Bessel function from the main lobe (at the horizon) inward through successive nulls and sidelobes toward transit. For $b_{\mathrm{ew}} = 15.17\;\mathrm{m}$ and $R \approx 16'$, the maximum sky-plane baseline at transit is $|u_{\mathrm{sky}}|_{\max} \approx 530\;\lambda$, giving a Bessel argument $2\pi |u| R \approx 15.5$ — between the **fourth and fifth Bessel zeros**. So the data should show four full Bessel oscillations (and four nulls) between sunrise and transit.

## 1.5b The modulating function: the lab-manual discrete-sum form

Section 1.5 derived the uniform-disk visibility as a jinc, $V(|u|)/V(0) = 2J_1(2\pi|u|R)/(2\pi|u|R)$. The AY 121 lab manual ([`src/ugradio/lab_interf/interf.tex`](../../../src/ugradio/lab_interf/interf.tex) §8) writes the *same* function under the name **modulating function** ($\mathrm{MF}_{\mathrm{theory}}$), using the 1-D brightness profile directly and a discrete-sum approximation to the integral:

$$
\mathrm{MF}_{\mathrm{theory}}(f_f R) \;=\; \frac{1}{R}\int_{-R}^{R} \sqrt{R^2 - \Delta h^2}\;\cos\!\bigl(2\pi\,f_f\,\Delta h\bigr)\,d\Delta h
\;\approx\; \delta h \sum_{n=-N}^{+N}\sqrt{1 - \!\left(\tfrac{n}{N}\right)^{\!2}}\;\cos\!\left(\frac{2\pi f_f R\,n}{N}\right).
$$

The crucial observation is that **the right-hand side depends only on the product $f_f R$**, not on $f_f$ and $R$ separately. Plot $\mathrm{MF}_{\mathrm{theory}}$ against the single dimensionless variable $f_f R$ (in wavelengths) and you obtain a universal curve whose zero crossings lie at the Bessel nulls $j_{1,k}/(2\pi) \approx \{0.6098,\,1.1166,\,1.6192,\,2.1205,\,\ldots\}$. Measuring where the observed modulating function crosses zero therefore measures $R$ directly, without any fit.

**Equivalence with the jinc *up to normalisation*.** The integral above is textbook-standard ($\int_{-R}^R \sqrt{R^2-x^2}\cos(\omega x)\,dx = \pi R\, J_1(\omega R)/\omega$; [B&W] §8.5), so

$$\mathrm{MF}^{\mathrm{lab}}_{\mathrm{theory}}(f_f R) \;=\; \frac{1}{R}\cdot\frac{\pi R\,J_1(2\pi f_f R)}{2\pi f_f} \;=\; \frac{J_1(2\pi f_f R)}{2 f_f}.$$

This is **proportional to**, but not equal to, the jinc $2J_1(2\pi f_f R)/(2\pi f_f R)$: in the limit $f_f \to 0$ the lab-manual form tends to $\pi R/2$ (it carries units of length), while the jinc tends to $1$. The two functions differ by an overall factor of $\pi R/2$ and therefore share the same zero-crossing positions $f_f R = j_{1,k}/(2\pi)$ — which is the only feature the §8 procedure relies on.

### Diameter from extrema positions (notebook 05 approach)

The lab's §8 procedure recovers $R$ from zero-crossing positions. Notebook 05 generalises this to use **all envelope extrema** — both nulls ($J_1$ zeros) and sidelobe peaks ($J_2$ zeros, see §1.7) — roughly doubling the number of independent constraints. The positions of extrema are equally amplitude-independent: they depend only on the Bessel argument $x = 2\pi q R$, not on the source flux, antenna gains, or limb-brightening profile.

For any identified extremum at projected baseline $q$ matched to the $k$-th Bessel root $j_k$:

$$R_k = \frac{j_k}{2\pi\,q_k^{\mathrm{obs}}},\qquad \hat R = \frac{1}{K}\sum_{k=1}^{K} R_k,\qquad \sigma_{\hat R} = \frac{\mathrm{std}(R_k)}{\sqrt{K}}.$$

This is the primary diameter measurement in notebook 05.

### Why zero-crossing and full-envelope fits disagree (a little)

Zero-crossing and full-envelope fits will in general give slightly different $R$: the former is insensitive to the amplitude calibration (only the *location* of zeros matters), whereas the latter uses every point including the sidelobes and is therefore sensitive to any multiplicative amplitude bias (chip gain, DC-correction leakage, limb brightening). If the two methods disagree by more than their combined statistical uncertainty, the excess is a diagnostic of one of those systematics. In notebook 05 we apply the chip-gain correction measured in 02a and propagate the DC-correction amplitude bias measured in 02b into the error budget; the two diameter estimates then agree to within their stated uncertainties.

---

## 1.6 Measuring the Solar Diameter

From the Bessel-function visibility, the $k$-th null occurs when

$$|u_{\mathrm{sky},k}|\,R = \frac{j_{1,k}}{2\pi}.$$

Given an observed null at sky-plane baseline $|u_{\mathrm{sky},k}|$ (in wavelengths), the angular radius is

$$\boxed{R = \frac{j_{1,k}}{2\pi\,|u_{\mathrm{sky},k}|}.}$$

(Bessel-zero values from [DLMF] §10.21 / [A&S] Table 9.5; the same diameter-from-null formula appears in the early radio-Sun observations of [Christiansen & Warburton 1955, Aust. J. Phys. 8, 474] and in the textbook treatment [TMS3] §13.1.)

Since $|u_{\mathrm{sky}}| \propto |\cos h|$, each null corresponds to a specific hour angle $h_k$ where $|\cos h_k|$ takes the appropriate value. Multiple nulls provide independent estimates whose scatter is a first-pass error bar.

A more precise estimate comes from a **nonlinear least-squares fit of the full Bessel envelope**, which uses every data point — not just the nulls — and gives a covariance matrix from the residuals. We report both. The formal covariance error is multiplied by $\sqrt{\chi^2_\nu}$ when $\chi^2_\nu > 1$, so that unmodelled systematics inflating the residuals are absorbed into the quoted uncertainty rather than ignored. The dominant terms in the diameter error budget are then:

| Source | Approx. fractional contribution | Reference |
|---|---|---|
| Statistical (envelope SNR + fit residuals) | $\sim 0.1\text{–}0.2\,\%$ | radiometer eq., [R&W] §4 / [TMS3] §6.2 |
| Baseline uncertainty $\sigma_{b}/b \sim 0.30/15.17$ | $\sim 2.0\,\%$ | lidar prior, `constants.py` |
| Limb brightening (uniform-disk bias) | $\sim 0.5\text{–}1\,\%$ | [Selhorst-modelling], [Alissandrakis-ALMA] |
| Bandwidth smearing of $|u|$ across analysis band | $\sim 0.3\,\%$ | [TMS3] §6.3 |
| Chip-to-chip gain step | $\lesssim 0.5\,\%$ | empirical, see nb 02a |

added in quadrature these give a realistic ~1.4 % total — i.e. ~0.45' on a ~32' diameter, with the lidar-baseline and limb-brightening systematics contributing comparably.

## 1.7 Sunspot Signatures in the Visibility

A sunspot is a localised region of enhanced (or, more commonly in radio, *suppressed*) emission offset from the disk centre by angular displacement $(\Delta\alpha,\,\Delta\delta)$. The radio behaviour of sunspots is mode-dependent: at 10 GHz the dominant emission above strong sunspot magnetic fields is **gyroresonance** at the second/third harmonic of the local cyclotron frequency $\nu_{\rm gyro} = eB/(2\pi m_e c)$, which can give either bright or dark spots depending on the line-of-sight magnetic geometry ([Dulk85] §IV; [BBG98] §3). The lab does not need to identify which mechanism is at work — only that the spot contributes a localised flux to the visibility.

By the linearity of the van Cittert–Zernike theorem ([TMS3] §14.1), the total visibility of disk + spot is

$$V_{\mathrm{total}} = (1 - f)\,V_{\mathrm{disk}} + f\,V_{\mathrm{spot}},$$

where $f$ is the fractional flux of the spot and $V_{\mathrm{spot}} = V_{\mathrm{pt}}\,e^{-2\pi i\,(u\,\Delta\alpha + v\,\Delta\delta)}$ is a point-source visibility with an additional phase from the spot's angular offset. The spot is treated as **unresolved** ($V_{\mathrm{pt}} \equiv 1$) — for a single 15.17 m baseline at $\lambda \approx 2.87\;\mathrm{cm}$, the fringe spacing is $\lambda/b_{\mathrm{ew}} \approx 6.5'$ (so the effective angular resolution is roughly half that, $\sim 3'$), and individual sunspot umbrae ($\sim 0.5'$, see [Stix] §8.2) are well below this. (A two-element interferometer does not strictly have a *synthesised* beam in the imaging sense — that would require multiple baselines — so the right comparison is fringe spacing vs feature size.)

**Key diagnostic — amplitude.** At the Bessel nulls of the disk, $V_{\mathrm{disk}} = 0$, so $|V_{\mathrm{total}}| \approx f$. A non-zero fringe amplitude at what should be a null is direct evidence of asymmetric structure — the **residual amplitude at the null directly measures the spot's flux fraction**:

$$f \approx \frac{|V_{\mathrm{null}}|}{|V(0)|}.$$

Caveats: (i) the *measured* envelope minimum does not sit exactly at the theoretical Bessel zero (because of finite cadence and the rapidly-changing $u$ near the null), so a small residual remains even with no spot; (ii) the formula assumes a single spot, whereas multiple spots add coherently with phase factors that can constructively or destructively interfere; (iii) limb brightening of the *disk* itself contributes a residual of similar magnitude — the apparent "$f$" is contaminated at the few-percent level (see Part V of nb 05 for the budget).

**Phase diagnostic (not implemented in this lab).** At a disk null, $V_{\mathrm{disk}} \approx 0$ so $V_{\mathrm{total}}(u) \approx f\,e^{-2\pi i\,u\,\Delta\alpha}$ (with $v\Delta\delta$ negligible since $v \approx 0$ for this dataset). The **slope of $\arg(V_{\mathrm{total}})$ vs $u$** in a small window around the null would directly give the **EW offset** of the spot from disk centre:

$$\Delta\alpha = -\frac{1}{2\pi}\,\frac{d\arg(V_{\mathrm{total}})}{du}.$$

This is the standard Fourier-imaging interpretation of a point source displaced from the phase centre ([TMS3] §3.1, eq. 3.7). The NS offset $\Delta\delta$ is **unconstrained** by a single-baseline EW interferometer. The amplitude diagnostic is used in notebook 05 (via the sunspot floor $f$ in the 3-parameter envelope fit); the phase diagnostic is left as a possible extension.

---

### Sidelobe peaks and the zeros of $J_2$

The nulls of the Bessel envelope $|2J_1(x)/x|$ provide the most prominent features for diameter measurement, but the **sidelobe peaks** between nulls are also at analytically known positions. Differentiating the jinc:

$$\frac{d}{dx}\left[\frac{J_1(x)}{x}\right] = \frac{J_0(x)}{x} - \frac{2J_1(x)}{x^2}.$$

Setting this to zero and applying the Bessel recurrence $J_2(x) = 2J_1(x)/x - J_0(x)$ shows that the extrema of $J_1(x)/x$ occur at the **zeros of $J_2$**: $j_{2,k} \approx 5.136, 8.417, 11.620, 14.796, \ldots$

Each sidelobe peak at observed baseline $|u_k|$ gives an independent radius estimate:

$$\boxed{R = \frac{j_{2,k}}{2\pi\,|u_{\mathrm{sky},k}|}.}$$

Combining $J_1$-zero (null) and $J_2$-zero (peak) measurements roughly doubles the number of independent constraints on the solar diameter from the same data.

### Physical meaning of the zero crossings

The nulls of the Bessel-function visibility have a simple physical interpretation: a null occurs **when an integral number of fringe periods fit exactly across the source diameter**.

At the $k$-th null, the projected baseline satisfies $|u| R = j_{1,k} / (2\pi)$, where $j_{1,k}$ is the $k$-th zero of the Bessel function $J_1$. Since the fringe spacing is $1/|u|$ (in angular units), the condition $|u| R \approx k/2$ means approximately $k$ half-fringes span the diameter $2R$. At these special spacings, the contributions from opposite sides of the source cancel exactly, and the net visibility vanishes.

This is why the null positions are such powerful diagnostics of source structure: each null directly measures the angular size through the ratio $R = j_{1,k} / (2\pi |u_k|)$, independent of the source flux, the antenna gains, or the system temperature.

### 1.6a Extracting the Bessel Envelope

The uniform-disk visibility at projected baseline $u$ is

$$V(h) = A(u)\,\frac{2J_1(2\pi|u|R)}{2\pi|u|R}\,e^{i\psi(h)} + n(h),$$

where $\psi(h) = 2\pi[Q_{\rm ew}\cos\delta\sin h + Q_{\rm ns}\sin L\cos\delta\cos h]$ is the geometric fringe phase that oscillates rapidly in $h$, and the Bessel jinc factor is the **slow envelope** we want to measure. The challenge is separating the slow envelope from the fast fringe.

#### Approach used in notebook 05

Notebook 05 extracts the envelope by taking the **amplitude** $|V(h)|$ of the complex visibility at each capture. Since $|V| = A(u)\,|2J_1(x)/x| + \text{noise}$, the magnitude directly gives the Bessel envelope without needing to model or remove the fast fringe oscillation. The per-capture amplitudes are then interpolated onto a uniform HA grid and smoothed with a running mean to suppress noise before peak-finding.

This approach is simple and robust: it requires no knowledge of the baseline (the fringe phase cancels in the magnitude), works per frequency channel, and is insensitive to cable-phase jumps between chips. The cost is that the running-mean smoothing introduces a small bias in the positions of sharp features (nulls), which is absorbed into the per-channel scatter of the extrema-based diameter estimate.

#### Alternative: matched-filter demodulation

When the baseline $(b_{\rm ew}, b_{\rm ns})$ is known from a prior calibration (notebook 04), an alternative is to multiply the visibility by $e^{-i\psi_{\rm model}(h)}$ to shift the fringe to baseband, then take the magnitude. This removes the fast oscillation exactly rather than averaging it out, at the cost of depending on the baseline calibration. This approach is not used in the current analysis but could be implemented as a cross-check.

## 1.7b Noise model and per-capture uncertainty

All five baseline-fitting methods (and the diameter / spot fits) need a per-capture uncertainty $\sigma$ on the complex visibility. We use two complementary estimates and take the larger:

1. **Per-channel scatter inside the analysis band.** For each capture $i$, the standard deviation of $|V_i(\nu)|$ across the $\sim 285$ good channels in the 70 MHz analysis band, divided by $\sqrt{N_{\mathrm{ch}}}$, gives an i.i.d.-channel estimate of the band-averaged $\sigma$. The correlator records this per-channel uncertainty in `corr_std`; we cross-check the two and they agree to factors of order unity.
2. **Off-fringe residual scatter.** After subtracting the best-fit fringe model from the data, the standard deviation of the residuals — in a small window of contiguous captures — provides a *post-fit* estimate that is insensitive to channel-to-channel coherence and naturally folds in non-Gaussian systematics.

The radiometer expectation for an adding interferometer with system temperature $T_{\mathrm{sys}}$, channel width $\Delta\nu$, and integration time $\Delta t$ is

$$\sigma_V \;\sim\; \frac{T_{\mathrm{sys}}}{\sqrt{\Delta\nu\,\Delta t}},$$

(the standard radiometer equation; [R&W] §4.1, eq. 4.13; [TMS3] §6.2, eq. 6.50). A quick consistency check (with $T_{\mathrm{sys}} \sim 100\;\mathrm{K}$ — the typical NCH X-band system temperature documented in [AY121-Lab3] — $\Delta\nu = 244\;\mathrm{kHz}$, $\Delta t \approx 2.5\;\mathrm{s}$, source flux $T_b \sim 10^4\;\mathrm{K}$ from [Zirin91]) gives an SNR per channel per capture of order $10^2$, which matches the empirical scatter.

When envelope smoothing is applied (e.g. in `extract_fringe_envelope`), the per-point sigmas are *inflated* by $\sqrt{N_{\mathrm{smooth}}}$ to account for the loss of independence between neighbouring smoothed samples. Without this correction, $\chi^2$-based covariance estimates collapse to artificially small uncertainties — this was the cause of the previously-reported "$\pm 0.00$ arcmin" diameter uncertainty (now fixed in `fit_solar_diameter_bessel`).

---

> **A note on references.** Every numerical claim, every named theorem, and every textbook technique used in this notebook is attributed to an entry in [`labs/03/REFERENCES.md`](../REFERENCES.md). Short keys like `[TMS3]`, `[VAL81]`, `[Selhorst04]`, `[BCD98]`, `[DLMF]` resolve to full bibliographic entries there. Where I am genuinely unsure of a specific paper's volume/page (cm-radio Sun radius is the most common case), I cite generically with "see e.g. ... and references therein" rather than fabricating a citation.

## 1.8 Baseline Determination: Seven Methods

The fringe phase (Section 1.2) encodes the baseline components $(b_{\mathrm{ew}}, b_{\mathrm{ns}})$. We now derive seven independent methods to extract them from the observed complex visibility $V(h) = |V|\,e^{i\phi(h)}$. Methods 1, 1a, 2 and 3 are linear-algebra-only techniques (no nonlinear solver, no initial guess); Methods 4, 4a and 5 are direct nonlinear fits of the forward model. Multiple independent methods are useful both as a cross-check on each other and as an inverse-variance-weighted combined estimator.

All methods start from the geometric delay at capture $i$:

$$\tau'_{g,i} = \frac{b_{\mathrm{ew}}}{c}\cos\delta_i\,\sin h_i + \frac{b_{\mathrm{ns}}}{c}\sin L\,\cos\delta_i\,\cos h_i,$$

where $\delta_i$ is the per-capture source declination (which varies slowly as the Sun moves). The baselines $b_{\mathrm{ew}}$ and $b_{\mathrm{ns}}$ are constant — only the geometry changes.

---

### Method 1: FFT in the x-coordinate

**Observable:** Complex visibility $V(h)$.
**Key idea:** The coordinate $x_i \equiv \cos\delta_i\,\sin h_i$ linearises the fringe phase for a pure EW baseline. (The $x$-substitution trick is described in [TMS3] §10.4 for single-baseline geometry calibration.)

Since $\phi = 2\pi(b_{\mathrm{ew}}/\lambda)\,x + (\text{terms involving } \cos h) + \phi_c$, the visibility (for $b_{\mathrm{ns}} \approx 0$) is a pure sinusoid in $x$ with spatial frequency $f_x = b_{\mathrm{ew}}/\lambda$.

**Baseline recovery:**

$$\boxed{b_{\mathrm{ew}} = |f_{x,\mathrm{peak}}| \times \frac{c}{\nu}.}$$

**Limitation:** Measures only $b_{\mathrm{ew}}$; cannot recover $b_{\mathrm{ns}}$.

---

### Method 1a: Short-time Fourier transform of the temporal fringe

**Observable:** Band-averaged complex visibility $V(h_s)$ as a *time series*.
**Key idea:** Slide a Hann-windowed segment of $N_{\mathrm{win}}$ captures along the visibility, take the FFT of each segment, and read off the **local fringe frequency** $\hat f_{f,i}$ at the window centre $h_i$. The set of measurements $\{(h_i, \hat f_{f,i})\}$ is then fit to the lab-manual fringe-frequency formula

$$\frac{f_f(h)}{\omega_\oplus} \;=\; \frac{b_{\mathrm{ew}}}{\lambda}\cos\delta\,\cos h \;-\; \frac{b_{\mathrm{ns}}}{\lambda}\sin L\,\cos\delta\,\sin h,$$

which is **linear in $(b_{\mathrm{ew}}, b_{\mathrm{ns}})$**. The least-squares solution is

$$\boxed{\;\begin{bmatrix} b_{\mathrm{ew}} \\ b_{\mathrm{ns}} \end{bmatrix} = (\mathbf{X}^T \mathbf{W}\mathbf{X})^{-1}\,\mathbf{X}^T \mathbf{W}\,\hat{\mathbf{f}}_f,\;}\qquad \mathbf{X}_i \;=\; \frac{\omega_\oplus\cos\delta_i}{\lambda}\bigl[\,\cos h_i,\;-\sin L\,\sin h_i\,\bigr].$$

This is the method the AY 121 lab manual ([interf.tex](../../../src/ugradio/lab_interf/interf.tex) §5.1, "Local Fringe Frequency") recommends as the **Week 1 sanity check** — *"Look at the Fourier transform of your data to check!"* — converted into a quantitative baseline estimator. Four implementation details matter for the NCH dataset and are documented in `utils/baseline_fitting.py::stft_fringe_frequency`:

1. **Per-chip operation.** The STFT runs independently on each chip so that no window straddles the inter-chip dead-time gaps; otherwise the FFT frequency axis would be corrupted by the missing samples.
2. **Uniform-HA resampling.** Inside each chip the captures are linearly interpolated onto a uniform hour-angle grid before the FFT runs. The NCH cadence has $\sim$40 % intra-chip jitter; on the raw samples the FFT bin width is biased and the recovered $\hat f_f$ drifts.
3. **SNR-weighted iterative outlier rejection.** Near the Bessel nulls of the disk envelope the fringe amplitude collapses below noise and the FFT peak finder occasionally locks onto a noise sidelobe. The fit is therefore done with weights $w_i \propto \mathrm{SNR}_i$ (so windows in the high-amplitude lobes between nulls drive the result) and then iterated with $3\sigma$ residual clipping until the surviving subset stabilises.
4. **Prior-constrained peak search.** The Bessel-null problem above is *also* attacked at its source by restricting the FFT peak search to a band of $\pm$``search_tol_hz`` around a *predicted* fringe frequency. The prediction is computed at each window centre from the lidar-prior baseline $(b_{\mathrm{ew}}^{\mathrm{prior}}, b_{\mathrm{ns}}^{\mathrm{prior}})$ via the §5.1 fringe-frequency formula, so the algorithm only ever picks peaks that lie within (say) 1 mHz of the physically expected location at that hour angle. This is what makes it possible to drop the SNR threshold from `min_snr=10` (a hard cut that discarded most off-transit windows) to `min_snr=3` (which keeps essentially the entire HA range) without descending into the wrong basin of the linear fit. The prior only restricts the *search range*; the linear least-squares fit of $f_f$ vs HA is still data-driven and recovers whatever baseline the data prefer inside the gate.

The conversion of the FFT frequency from cycles-per-radian-of-HA to Hz uses $f_f^{\mathrm{Hz}} = f_f^{\mathrm{cyc/rad}}\cdot\omega_\oplus$, which avoids any dependence on the (inaccurate) median capture cadence.

**Strength:** A purely *temporal* measurement, completely independent of the absolute fringe phase or the cable delay. It is the most pedagogically transparent method — every step (FFT, peak-find, linear fit) is by hand visible — and the residuals plot in nb 04 §4.1a is a direct visual demonstration that the data obeys the lab manual's fringe-frequency formula.

**Limitation:** Each window measurement has a precision floor set by the FFT bin width $\Delta f \sim 1/(N_{\mathrm{win}}\,\Delta h)$ and by *chirp smearing* (the fringe frequency is itself a function of $h$, so a finite window contains a spread of true $f_f$ values). A larger window improves SNR per measurement but increases chirp smearing; the chosen $N_{\mathrm{win}} = 64$ captures (~80 s, with `step_size=16`) is the smallest window that still keeps the per-measurement precision below the linear-fit residual scatter, while doubling the number of windows that survive the prior-constrained gate compared to $N_{\rm win}=128$. The resulting per-window precision is ~0.4 mHz on the band-averaged fringe and the surviving 73 windows span essentially the full observation range from $h \approx -69°$ to $h \approx +20°$.

---

### Method 2: Phase slope

**Observable:** Unwrapped phase $\phi(h) = \arg V(h)$.
**Key idea:** Fit the per-capture phase using the physical regressors that encode the per-capture geometry. (Standard linear-regression interferometric calibration; [TMS3] §10.1.)

The design matrix is $\mathbf{X} = [\cos\delta_i\sin h_i,\;\sin L\,\cos\delta_i\cos h_i,\;\mathbf{1}]$, and the fit gives:

$$\phi_i = A\,\cos\delta_i\sin h_i + B\,\sin L\,\cos\delta_i\cos h_i + \phi_0,$$

where $A = 2\pi b_{\mathrm{ew}}/\lambda$ and $B = 2\pi b_{\mathrm{ns}}/\lambda$, so:

$$\boxed{b_{\mathrm{ew}} = \frac{A\,c}{2\pi\nu}, \qquad b_{\mathrm{ns}} = \frac{B\,c}{2\pi\nu}.}$$

Note: because the per-capture $\cos\delta_i$ is folded into the regressors, the baseline recovery is a simple scaling — no division by $\cos\delta$ is needed.

**Uncertainty:** $\mathrm{Cov}(\hat{\boldsymbol{\beta}}) = \hat{\sigma}^2\,(\mathbf{X}^T\mathbf{X})^{-1}$ ([Bevington & Robinson 2003] / [AY121-FitNotes]).

---

### Method 3: Lag-spectrum delay

**Observable:** Complex cross-spectrum $V(\nu, h)$ across frequency channels.
**Key idea:** IFFT gives the geometric delay $\tau_g$ per capture (this is the standard "delay calibration" approach used in connected-element arrays such as the VLA; [TMS3] §10.2); fit these delays using the same physical regressors:

$$\tau_i = b_{\mathrm{ew}}\,\frac{\cos\delta_i\,\sin h_i}{c} + b_{\mathrm{ns}}\,\frac{\sin L\,\cos\delta_i\,\cos h_i}{c} + \tau_{\mathrm{inst}}.$$

**Baseline recovery:** The fit coefficients directly give $b_{\mathrm{ew}}$ and $b_{\mathrm{ns}}$ (after unit conversion from nanoseconds to metres).

---

### Method 4: Nonlinear least-squares (NLS) complex fringe fit

**Observable:** Full complex visibility $V(h)$ (Re + Im).
**Key idea:** Fit the fringe equation directly, with $(b_{\mathrm{ew}}, b_{\mathrm{ns}})$ as nonlinear parameters and $(A, B, C, D)$ as linear parameters solved analytically at each step (the standard separable-variable trick of [Golub & Pereyra 1973, SIAM J. Numer. Anal. 10, 413]).

The model uses per-capture declination:

$$\psi_i = \frac{2\pi}{\lambda}\bigl[b_{\mathrm{ew}}\cos\delta_i\sin h_i + b_{\mathrm{ns}}\sin L\,\cos\delta_i\cos h_i\bigr],$$

$$V_i = (A + iC)\cos\psi_i + (B + iD)\sin\psi_i.$$

The optimizer searches over $(b_{\mathrm{ew}}, b_{\mathrm{ns}})$ in metres, seeded from the phase-slope result. Because the cost function is fitted to **both** Re($V$) and Im($V$) — $2N$ data points for $N$ captures — this is the most statistically efficient single-channel estimator: with the same data, its formal uncertainty is $\sim\!\sqrt{2}$ smaller than the real-only Method 4a below.

---

### Method 4a: NLS *real-only* fringe fit — the lab-manual prescription

**Observable:** **Real part only** of the complex visibility, $\mathrm{Re}\,V(h)$.
**Key idea:** Fit the lab manual's boxed fringe response equation ([interf.tex](../../../src/ugradio/lab_interf/interf.tex), eq. labelled `fringeresponse`) literally:

$$\boxed{\;F(h_s) \;=\; A\cos\!\bigl(2\pi\nu\,\tau'_g(h_s)\bigr) \;+\; B\sin\!\bigl(2\pi\nu\,\tau'_g(h_s)\bigr).\;}$$

The nonlinear parameters are $(b_{\mathrm{ew}}, b_{\mathrm{ns}})$ and the linear parameters $(A, B)$ are again solved by Golub–Pereyra projection at each step. The cost function is now $N$ data points (Re $V$ only) instead of $2N$, so the formal uncertainty is roughly $\sqrt{2}$ larger than Method 4 at fixed signal-to-noise.

**Why we keep this method as well as Method 4.** The lab manual's adding-interferometer derivation in §4–5 of `interf.tex` produces a *real* fringe pattern $F(h_s)$ — there is no imaginary part in the analog signal chain, since the front end is a single mixer/detector. The complex visibility we actually record on the SNAP correlator's FX backend is a *correlation* of two voltages, which is conceptually different from the §4 fringe (the real-and-imaginary structure of the recorded $V$ comes from the correlator's quadrature sampling, not from an intrinsic complex sky brightness). The lab-manual prescription is therefore "real-only" by construction. Method 4 (complex) is the natural fit for our digital correlator data; Method 4a is the literal lab-manual reading. They should agree, and the comparison is itself a useful cross-check on whether the imaginary part of $V$ is doing what the simple real-fringe model predicts. In practice the two recover the same baseline to $\lesssim 1$ cm; the residual difference goes into the inter-method scatter that inflates the adopted uncertainty in nb 04 §4.7.

---

### Method 5: Brute-force grid search

**Observable:** Same as NLS (full complex visibility).
**Key idea:** Evaluate the NLS cost function on a 2-D grid over $(Q_{\mathrm{ew}}, Q_{\mathrm{ns}}) = (b_{\mathrm{ew}}/\lambda,\; b_{\mathrm{ns}}/\lambda)$, solving $(A, B, C, D)$ analytically at each grid point. Per-capture $\cos\delta_i$ enters the phase computation at every point.

Two passes: a **coarse** grid to locate the global minimum among the many sidelobes, then a **fine** grid to refine the position and compute the curvature matrix $[\alpha]$ and covariance $[\alpha]^{-1}$ for uncertainties (the Bevington-style $\chi^2$ procedure of [Bevington & Robinson 2003] §11; documented as the lab's standard fitting method in [AY121-FitNotes]). This is the procedure prescribed step-by-step in [`fitting_notes_2017.pdf`](../../../src/ugradio/lab_interf/fitting_notes_2017.pdf); §1.2b above derives why it works.

The lab manual recommends using the brute-force result as the *initial guess* for a "proper" Levenberg–Marquardt nonlinear fit and comparing the two; nb 04 §4.6b runs that comparison.

---

### Summary

| Method | Observable | Recovers $b_{\mathrm{ns}}$? | Phase unwrapping? | Initial guess? | Reference |
|--------|-----------|:-:|:-:|:-:|---|
| 1 — FFT in $x$-space | Complex $V$ remapped to $x$-axis | No | No | No | [TMS3] §10.4 |
| 1a — STFT of temporal fringe | Time-windowed FFT of $V(h)$ → $f_f(h)$ | Yes | No | No | [interf.tex] §5.1 |
| 2 — Phase slope | $\arg V(h)$ | Yes | Yes | No | [TMS3] §10.1 |
| 3 — Lag-spectrum delay | $\mathrm{IFFT}_\nu V(\nu, h)$ | Yes | No | No | [TMS3] §10.2 |
| 4 — NLS complex | Re + Im of $V$ | Yes | No | Yes (phase slope) | [Golub & Pereyra 1973] |
| 4a — NLS real | Re $V$ only (lab-manual form) | Yes | No | Yes (phase slope) | [interf.tex] §4–5, eq. `fringeresponse` |
| 5 — Brute-force grid + curvature | Re + Im of $V$ | Yes | No | No (exhaustive) | [Bevington & Robinson 2003] §11; [AY121-FitNotes] |

All seven recoveries are run side-by-side in nb 04 and combined into a single inverse-variance-weighted estimator at the end of §4.7.


## What this notebook supplies to notebooks 03-05

- **Notebook 03** uses Sections 1.3 and 1.5 to predict the fringe-frequency band, the transit fringe period, and the Bessel-null locations before any fitting is attempted.
- **Notebook 04** uses Sections 1.2b and 1.8 for the STFT and brute-force baseline fits, including the lab-manual $Q_{\mathrm{ew}}/Q_{\mathrm{ns}}$ notation and the distinction between required methods and cross-checks.
- **Notebook 05** uses Sections 1.5-1.7 for the diameter fit, the modulating-function interpretation, and the visibility signature of a localised active region on top of the solar disk.

If a later notebook introduces a quantity without re-deriving it, this is the notebook that defines it.
